# Amazon Fashion Dataset Preprocessing

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

## 1. Download Dataset

In [4]:
URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Amazon_Fashion.csv.gz"

df = pd.read_csv(URL, compression='gzip')
print(f"Columns: {df.columns.tolist()}")
print(f"Raw data: {len(df):,} interactions")

df = df.rename(columns={'parent_asin': 'item_id'})
print(f"Users: {df['user_id'].nunique():,} | Items: {df['item_id'].nunique():,}")
df.head()

Columns: ['user_id', 'parent_asin', 'rating', 'timestamp']
Raw data: 2,474,375 interactions
Users: 2,035,490 | Items: 825,869


,user_id,item_id,rating,timestamp
0,AGBFYI2DDIKXC5Y4FARTYDTQBMFQ,B00LOPVX74,5.0,1578528394489
1,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B07B4JXK8D,5.0,1608426246701
2,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B007ZSEQ4Q,2.0,1432344828000
3,AFVNEEPDEIH5SPUN5BWC6NKL3WNQ,B07F2BTFS9,1.0,1546289847095
4,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,B00XESJTDE,5.0,1439476166000


## 2. Filter Rating >= 3.5

In [5]:
df_filtered = df[df['rating'] >= 3.5].copy()
print(f"After rating filter: {len(df_filtered):,} interactions")
print(f"Users: {df_filtered['user_id'].nunique():,} | Items: {df_filtered['item_id'].nunique():,}")

After rating filter: 1,759,664 interactions
Users: 1,490,952 | Items: 637,129


## 3. K-Core Filtering (k=3)

In [6]:
def k_core_filter(df, k=3):
    prev_len = 0
    iteration = 0
    while len(df) != prev_len:
        prev_len = len(df)
        iteration += 1
        
        user_counts = df['user_id'].value_counts()
        valid_users = user_counts[user_counts >= k].index
        df = df[df['user_id'].isin(valid_users)]
        
        item_counts = df['item_id'].value_counts()
        valid_items = item_counts[item_counts >= k].index
        df = df[df['item_id'].isin(valid_items)]
        
        print(f"Iteration {iteration}: {len(df):,} interactions | Users: {df['user_id'].nunique():,} | Items: {df['item_id'].nunique():,}")
    
    return df

df_core = k_core_filter(df_filtered, k=3)
print(f"\nFinal: {len(df_core):,} interactions | Users: {df_core['user_id'].nunique():,} | Items: {df_core['item_id'].nunique():,}")

Iteration 1: 48,993 interactions | Users: 27,175 | Items: 9,310
Iteration 2: 12,727 interactions | Users: 3,980 | Items: 2,487
Iteration 3: 8,583 interactions | Users: 1,910 | Items: 1,664
Iteration 4: 7,531 interactions | Users: 1,476 | Items: 1,475
Iteration 5: 7,220 interactions | Users: 1,366 | Items: 1,415
Iteration 6: 7,074 interactions | Users: 1,308 | Items: 1,395
Iteration 7: 7,015 interactions | Users: 1,290 | Items: 1,383
Iteration 8: 6,980 interactions | Users: 1,279 | Items: 1,374
Iteration 9: 6,973 interactions | Users: 1,275 | Items: 1,374
Iteration 10: 6,973 interactions | Users: 1,275 | Items: 1,374

Final: 6,973 interactions | Users: 1,275 | Items: 1,374


## 4. Label Encoding

In [7]:
le_user = LabelEncoder()
le_item = LabelEncoder()

df_core['user_id'] = le_user.fit_transform(df_core['user_id'])
df_core['item_id'] = le_item.fit_transform(df_core['item_id'])

print(f"User ID range: 0 - {df_core['user_id'].max()}")
print(f"Item ID range: 0 - {df_core['item_id'].max()}")
df_core.head()

User ID range: 0 - 1274
Item ID range: 0 - 1373


,user_id,item_id,rating,timestamp
28,559,383,5.0,1549403115093
30,559,447,5.0,1561338044117
32,559,900,5.0,1579433019201
33,559,1002,5.0,1587205458621
176,1017,366,4.0,1539698338504


## 5. Export to CSV

In [8]:
output_path = Path("interaction.csv")
df_core[['user_id', 'item_id']].to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.2f} KB")

Saved to: interaction.csv
File size: 62.88 KB


## 6. Verify

In [9]:
df_check = pd.read_csv(output_path)
n_users = df_check['user_id'].nunique()
n_items = df_check['item_id'].nunique()
n_interactions = len(df_check)
sparsity = 1 - n_interactions / (n_users * n_items)

print(f"# Users: {n_users:,}")
print(f"# Items: {n_items:,}")
print(f"# Interactions: {n_interactions:,}")
print(f"Sparsity: {sparsity:.6f} ({sparsity*100:.4f}%)")
print(f"\nMin interactions per user: {df_check['user_id'].value_counts().min()}")
print(f"Min interactions per item: {df_check['item_id'].value_counts().min()}")
df_check.head()

# Users: 1,275
# Items: 1,374
# Interactions: 6,973
Sparsity: 0.996020 (99.6020%)

Min interactions per user: 3
Min interactions per item: 3


,user_id,item_id
0,559,383
1,559,447
2,559,900
3,559,1002
4,1017,366
